In [ ]:
!pip install gymnasium[box2d]

In [ ]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import gymnasium as gym

In [ ]:
@dataclass
class Args:
    """SAC hyperparameters"""
    num_envs: int = 1
    total_timesteps: int = 200_000
    learning_rate: float = 3e-4
    buffer_size: int = 1_000_000
    gamma: float = 0.99
    tau: float = 0.005
    batch_size: int = 256
    update_every: int = 50
    update_after: int = 1_000
    start_steps: int = 10_000
    target_entropy_scale: float = 0.98
    hidden_sizes: tuple = (256, 256)
    num_minibatches: int = 1
    update_epochs: int = 1


args = Args()

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
LOG_STD_MIN = -20
LOG_STD_MAX = 2


def mlp(sizes, activation=nn.ReLU, output_activation=nn.Identity):
    """Create a multi-layer perceptron"""
    layers = []
    for i in range(len(sizes) - 1):
        act = activation if i < len(sizes) - 2 else output_activation
        layers += [nn.Linear(sizes[i], sizes[i + 1]), act()]
    return nn.Sequential(*layers)

In [ ]:
class SquashedGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_sizes, act_limit):
        super().__init__()
        self.net = mlp([obs_dim] + list(hidden_sizes), nn.ReLU, nn.ReLU)
        self.mu_layer = nn.Linear(hidden_sizes[-1], act_dim)
        self.log_std_layer = nn.Linear(hidden_sizes[-1], act_dim)
        self.act_limit = act_limit

    def forward(self, obs, deterministic=False, with_logprob=True):
        h = self.net(obs)
        mu = self.mu_layer(h)
        log_std = self.log_std_layer(h)
        log_std = torch.clamp(log_std, LOG_STD_MIN, LOG_STD_MAX)
        std = torch.exp(log_std)
        
        dist = torch.distributions.Normal(mu, std)
        
        if deterministic:
            pre_tanh = mu
        else:
            pre_tanh = dist.rsample()  # reparameterization trick
        
        if with_logprob:
            logp = dist.log_prob(pre_tanh).sum(dim=-1, keepdim=True)
            # tanh correction
            logp -= (2 * (math.log(2) - pre_tanh - F.softplus(-2 * pre_tanh))).sum(dim=-1, keepdim=True)
        else:
            logp = None
        
        action = torch.tanh(pre_tanh)
        action = self.act_limit * action
        return action, logp


class QFunction(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_sizes):
        super().__init__()
        self.q = mlp([obs_dim + act_dim] + list(hidden_sizes) + [1], nn.ReLU, nn.Identity)
    
    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q(x)

In [ ]:
class SACAgent:
    def __init__(
        self,
        obs_dim,
        act_dim,
        act_limit,
        args,
    ):
        self.args = args
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.act_limit = act_limit
        self.device = device
        
        self.actor = SquashedGaussianActor(obs_dim, act_dim, args.hidden_sizes, act_limit).to(device)
        self.q1 = QFunction(obs_dim, act_dim, args.hidden_sizes).to(device)
        self.q2 = QFunction(obs_dim, act_dim, args.hidden_sizes).to(device)
        
        self.q1_target = QFunction(obs_dim, act_dim, args.hidden_sizes).to(device)
        self.q2_target = QFunction(obs_dim, act_dim, args.hidden_sizes).to(device)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())
        
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=args.learning_rate)
        self.q1_opt = optim.Adam(self.q1.parameters(), lr=args.learning_rate)
        self.q2_opt = optim.Adam(self.q2.parameters(), lr=args.learning_rate)
        
        # Automatic temperature tuning
        self.target_entropy = -act_dim * args.target_entropy_scale
        self.log_alpha = torch.tensor(0.0, requires_grad=True, device=device)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=args.learning_rate)
    
    @property
    def alpha(self):
        return self.log_alpha.exp()
    
    def act(self, obs, deterministic=False):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            action, _ = self.actor(obs_t, deterministic=deterministic, with_logprob=False)
        return action.cpu().numpy()[0]
    
    def update(self, batch):
        obs = batch["obs"]
        act = batch["act"]
        rew = batch["rew"]
        next_obs = batch["next_obs"]
        done = batch["done"]
        
        # -------------------------
        # Q update
        # -------------------------
        with torch.no_grad():
            next_act, next_logp = self.actor(next_obs, deterministic=False, with_logprob=True)
            q1_target_next = self.q1_target(next_obs, next_act)
            q2_target_next = self.q2_target(next_obs, next_act)
            q_target_next = torch.min(q1_target_next, q2_target_next) - self.alpha * next_logp
            backup = rew + self.args.gamma * (1.0 - done) * q_target_next
        
        q1_pred = self.q1(obs, act)
        q2_pred = self.q2(obs, act)
        
        q1_loss = F.mse_loss(q1_pred, backup)
        q2_loss = F.mse_loss(q2_pred, backup)
        q_loss = q1_loss + q2_loss
        
        self.q1_opt.zero_grad()
        self.q2_opt.zero_grad()
        q_loss.backward()
        self.q1_opt.step()
        self.q2_opt.step()
        
        # -------------------------
        # Policy update
        # -------------------------
        for p in self.q1.parameters():
            p.requires_grad = False
        for p in self.q2.parameters():
            p.requires_grad = False
        
        pi, logp_pi = self.actor(obs, deterministic=False, with_logprob=True)
        q1_pi = self.q1(obs, pi)
        q2_pi = self.q2(obs, pi)
        q_pi = torch.min(q1_pi, q2_pi)
        
        actor_loss = (self.alpha * logp_pi - q_pi).mean()
        
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()
        
        for p in self.q1.parameters():
            p.requires_grad = True
        for p in self.q2.parameters():
            p.requires_grad = True
        
        # -------------------------
        # Temperature update
        # -------------------------
        alpha_loss = -(self.log_alpha * (logp_pi + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()
        
        # -------------------------
        # Target update
        # -------------------------
        with torch.no_grad():
            for p, p_targ in zip(self.q1.parameters(), self.q1_target.parameters()):
                p_targ.data.mul_(1 - self.args.tau)
                p_targ.data.add_(self.args.tau * p.data)
            
            for p, p_targ in zip(self.q2.parameters(), self.q2_target.parameters()):
                p_targ.data.mul_(1 - self.args.tau)
                p_targ.data.add_(self.args.tau * p.data)
        
        return {
            "q_loss": q_loss.item(),
            "actor_loss": actor_loss.item(),
            "alpha_loss": alpha_loss.item(),
            "alpha": self.alpha.item(),
            "q1": q1_pred.mean().item(),
            "q2": q2_pred.mean().item(),
        }
    
    def save(self, path='sac_lunar_lander.pth'):
        checkpoint = {
            'actor_state_dict': self.actor.state_dict(),
            'q1_state_dict': self.q1.state_dict(),
            'q2_state_dict': self.q2.state_dict(),
            'log_alpha': self.log_alpha,
            'actor_opt_state_dict': self.actor_opt.state_dict(),
            'q1_opt_state_dict': self.q1_opt.state_dict(),
            'q2_opt_state_dict': self.q2_opt.state_dict(),
            'alpha_opt_state_dict': self.alpha_opt.state_dict(),
        }
        torch.save(checkpoint, path)
        print(f"Model saved to {path}")

In [ ]:
class ReplayBuffer:
    def __init__(self, obs_dim, act_dim, size):
        self.obs = np.zeros((size, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((size, obs_dim), dtype=np.float32)
        self.acts = np.zeros((size, act_dim), dtype=np.float32)
        self.rews = np.zeros((size, 1), dtype=np.float32)
        self.done = np.zeros((size, 1), dtype=np.float32)
        self.ptr = 0
        self.size = 0
        self.max_size = size

    def add(self, obs, act, rew, next_obs, done):
        self.obs[self.ptr] = obs
        self.acts[self.ptr] = act
        self.rews[self.ptr] = rew
        self.next_obs[self.ptr] = next_obs
        self.done[self.ptr] = done
        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample(self, batch_size, device):
        idxs = np.random.randint(0, self.size, size=batch_size)
        batch = dict(
            obs=torch.as_tensor(self.obs[idxs], device=device),
            act=torch.as_tensor(self.acts[idxs], device=device),
            rew=torch.as_tensor(self.rews[idxs], device=device).squeeze(1),
            next_obs=torch.as_tensor(self.next_obs[idxs], device=device),
            done=torch.as_tensor(self.done[idxs], device=device).squeeze(1),
        )
        return batch

In [ ]:
def evaluate(agent: SACAgent, env_name: str, episodes=5, seed=42):
    """Evaluate agent performance over multiple episodes"""
    env = gym.make(env_name)
    scores = []
    
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        done = False
        truncated = False
        ep_ret = 0.0
        
        while not (done or truncated):
            action = agent.act(obs, deterministic=True)
            obs, reward, done, truncated, _ = env.step(action)
            ep_ret += reward
        
        scores.append(ep_ret)
    
    env.close()
    return float(np.mean(scores))

In [ ]:
def train_sac():
    set_seed(42)
    
    env = gym.make('LunarLanderContinuous-v3')
    
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]
    act_limit = float(env.action_space.high[0])
    
    agent = SACAgent(
        obs_dim=obs_dim,
        act_dim=act_dim,
        act_limit=act_limit,
        args=args,
    )
    
    replay = ReplayBuffer(obs_dim, act_dim, size=args.buffer_size)
    
    num_updates = args.total_timesteps // args.update_every
    
    # History tracking
    history = {
        'step': [], 'reward': [],
        'q_loss': [], 'actor_loss': [], 'alpha_loss': [], 'alpha': []
    }
    
    print(f"Starting training on {device}...")
    
    obs, _ = env.reset(seed=42)
    episode_return = 0.0
    episode_len = 0
    global_step = 0
    
    for update in range(1, num_updates + 1):
        # Collect experience
        for _ in range(args.update_every):
            global_step += 1
            
            if global_step < args.start_steps:
                action = env.action_space.sample()
            else:
                action = agent.act(obs, deterministic=False)
            
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_return += reward
            episode_len += 1
            
            # Truncation is time-limit, not true terminal
            done_for_buffer = float(terminated)
            replay.add(obs, action, reward, next_obs, done_for_buffer)
            obs = next_obs
            
            if done or (episode_len == 1000):
                # Record episode metrics
                history['step'].append(global_step)
                history['reward'].append(episode_return)
                
                print(f"Update {update}/{num_updates} - Step {global_step} - Reward: {episode_return:.2f}")
                
                obs, _ = env.reset()
                episode_return = 0.0
                episode_len = 0
        
        # Train
        q_losses = []
        actor_losses = []
        alpha_losses = []
        alphas = []
        
        for _ in range(args.update_every):
            batch = replay.sample(args.batch_size, device)
            info = agent.update(batch)
            q_losses.append(info["q_loss"])
            actor_losses.append(info["actor_loss"])
            alpha_losses.append(info["alpha_loss"])
            alphas.append(info["alpha"])
        
        # Save history
        history['q_loss'].append(np.mean(q_losses))
        history['actor_loss'].append(np.mean(actor_losses))
        history['alpha_loss'].append(np.mean(alpha_losses))
        history['alpha'].append(np.mean(alphas))
        
        # Save checkpoint periodically
        if update % 50 == 0:
            agent.save(f'sac_checkpoint_{update}.pth')
    
    env.close()
    agent.save('sac_final.pth')
    return agent, history

In [ ]:
# Train the model
trained_agent, training_history = train_sac()

In [ ]:
import matplotlib.pyplot as plt

def plot_training_results(history):
    """Plot training metrics with twin axes"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Episode Reward
    ax1.plot(history['step'], history['reward'], color='tab:blue', alpha=0.6, label='Reward')
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Episode Reward')
    ax1.set_title('Training Episode Returns')
    ax1.grid(True, alpha=0.3)
    
    # Q-Loss
    ax2.plot(history['step'][:len(history['q_loss'])], history['q_loss'], color='tab:red', label='Q-Loss')
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Loss')
    ax2.set_title('Q-Function Loss')
    ax2.grid(True, alpha=0.3)
    
    # Actor Loss
    ax3.plot(history['step'][:len(history['actor_loss'])], history['actor_loss'], color='tab:green', label='Actor Loss')
    ax3.set_xlabel('Steps')
    ax3.set_ylabel('Loss')
    ax3.set_title('Actor Policy Loss')
    ax3.grid(True, alpha=0.3)
    
    # Alpha
    ax4.plot(history['step'][:len(history['alpha'])], history['alpha'], color='tab:orange', label='Alpha')
    ax4.set_xlabel('Steps')
    ax4.set_ylabel('Value')
    ax4.set_title('Alpha (Temperature)')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_results(training_history)

In [ ]:
import imageio
from IPython.display import Image, display

def record_gif(checkpoint_path, filename='simulation.gif'):
    """Load a model from file and record a simulation GIF."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Reconstruct environment parameters
    temp_env = gym.make('LunarLanderContinuous-v3')
    obs_dim = temp_env.observation_space.shape[0]
    act_dim = temp_env.action_space.shape[0]
    act_limit = float(temp_env.action_space.high[0])
    temp_env.close()
    
    # Initialize Agent
    agent = SACAgent(
        obs_dim=obs_dim,
        act_dim=act_dim,
        act_limit=act_limit,
        args=args,
    )
    
    # Load the checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    agent.actor.load_state_dict(checkpoint['actor_state_dict'])
    agent.q1.load_state_dict(checkpoint['q1_state_dict'])
    agent.q2.load_state_dict(checkpoint['q2_state_dict'])
    if checkpoint['log_alpha'] is not None:
        agent.log_alpha.data.copy_(checkpoint['log_alpha'].data)
    
    env = gym.make('LunarLanderContinuous-v3', render_mode='rgb_array')
    obs, _ = env.reset(seed=42)
    frames = []
    done = False
    truncated = False
    total_reward = 0
    
    print(f"Simulating using {checkpoint_path}...")
    while not (done or truncated):
        frames.append(env.render())
        # Use the agent's act method for continuous actions
        action = agent.act(obs, deterministic=True)
        obs, reward, done, truncated, _ = env.step(action)
        total_reward += reward
    
    env.close()
    print(f"Total Reward: {total_reward:.2f}")
    imageio.mimsave(filename, frames, fps=30)
    return filename

In [ ]:
# Example usage: recording from the final model file
gif_path = record_gif('sac_final.pth')
display(Image(filename=gif_path))

In [ ]:
# Example usage: recording from a checkpoint
gif_path = record_gif('sac_checkpoint_500.pth')
display(Image(filename=gif_path))